# Neural Bandit Algorithm Evaluation Framework

## Setup: Paper 8 (RL/Q-learning) External Testbed — Paper Defaults

This notebook runs the Paper 8 external testbed **using the paper's own topology/physics configuration first** (paper-config-first).

- Paper-config-first notebook: `H-MABs_Eval-Testbed-Paper8-PaperRunConfig.ipynb` (this file)
- Standardized sweep (run later): `H-MABs_Eval-Testbed-Paper8-StandardizedRunConfig.ipynb`


## Environment Setup & Library Installation


In [ ]:
# ============================================================
# Setup: Quantum MAB Framework (Paper 8 RL/Q-learning Testbed)
# ============================================================

# --- (Optional) Install dependencies (Colab/local) ---
# !pip install -q torch torchvision numpy matplotlib seaborn pandas tqdm scipy scikit-learn pmdarima networkx

# --- Core Imports ---
import os, sys, warnings, importlib
import numpy as np
import networkx as nx
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Path Setup ---
print(f"Current working directory: {os.getcwd().split('/')[-1]}")
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    project_dir = '/content/drive/MyDrive/GA-Work/hybrid_variable_framework/Dynamic_Routing_Eval_Framework'
    os.chdir(project_dir)
    print("Running in Google Colab")
except ImportError:
    print("Running locally (not in Colab)")
    current_dir = os.getcwd()
    if 'GA-Work' in current_dir and 'Dynamic_Routing_Eval_Framework' not in current_dir:
        candidate = os.path.join(current_dir, 'hybrid_variable_framework', 'Dynamic_Routing_Eval_Framework')
        if os.path.exists(candidate):
            os.chdir(candidate)
            print(f"Changed to project directory: {os.getcwd().split('/')[-1]}")

# Add necessary paths for daqr package discovery
sys.path.append(os.path.join(os.getcwd(), 'src'))
sys.path.append(os.getcwd())

import daqr
print('✓ daqr import OK')

from daqr.config.experiment_config import ExperimentConfiguration
from daqr.evaluation.allocator_runner import AllocatorRunner

from daqr.core.topology_generator import Paper8RandomConnectedTopologyGenerator
from daqr.core.quantum_physics import Paper8NoiseModel, Paper8FidelityCalculator

# (Optional) reload core modules during dev
from daqr.core import topology_generator, quantum_physics
importlib.reload(topology_generator)
importlib.reload(quantum_physics)
print('✓ core modules loaded')


## Paper 8 Run Configuration (Paper Defaults First)


In [ ]:
# --- Paper 8 run configuration (paper defaults first) ---
config = ExperimentConfiguration()
models = config.NEURAL_MODELS  # keep consistent with other testbed notebooks

# Paper 8 upstream simulation settings (reference only)
PAPER8_N_ITERATIONS = 1000
PAPER8_EPISODES_PER_AGENT = 15000
PAPER8_MODES = [1, 2, 3]

# Framework run settings for this paper-config notebook
# (Standardized run-config sweep comes later in the other notebook.)
BASE_FRAMES = PAPER8_N_ITERATIONS
FRAME_STEP  = PAPER8_N_ITERATIONS
RUNS        = [1]
SCALES       = [1]
ALLOCATORS   = ['Default']

ATTACK_INTENSITY = 0.25

test_scenarios = {
    'stochastic': 'Stochastic Environment (Natural Network Conditions)',
    'none': 'Baseline (Optimal Conditions)',
}

FRAMEWORK_CONFIG = {
    'exp_num': 1,
    'test_mode': True,
    'base_frames': BASE_FRAMES,
    'frame_step': FRAME_STEP,
    'models': models,
    'intensity': ATTACK_INTENSITY,
    'routing_strategy': 'fixed',
    'capacity': 10000,
    'main_env': 'stochastic',
    'env_attrs': {
        'intensity': ATTACK_INTENSITY,
        'base_seed': 10,  # Paper 8 sets numpy/random seed=10
        'reproducible': True,
    },

    # Paper 8 configuration (from upstream [R]Qsim-top-V6-Resource.ipynb)
    'paper8': {
        'testbed': 'paper8',
        'num_paths': 8,
        'total_qubits': 35,
        'min_qubits_per_route': 2,

        # topology parameters
        'num_nodes': 20,
        'connection_prob': 0.001,
        'fidelity_range': (0.65, 0.99),
        'rate_range': (0.75, 1.0),
        'pur_round_range': (0, 3),
        'swap_success_range': (0.23, 0.8),

        # physics params
        'min_fidelity': 0.0,
    },
}

print('BASE_FRAMES:', BASE_FRAMES)
print('FRAME_STEP:', FRAME_STEP)
print('RUNS:', RUNS)
print('SCALES:', SCALES)
print('ALLOCATORS:', ALLOCATORS)
print('PHYSICS_MODELS: paper8')


## Paper 8 Helper Functions (paths, contexts, physics adapter)


In [ ]:
# --- Paper 8 helper functions + physics adapter ---
import itertools

def generate_paper8_paths(topology: nx.Graph, num_paths: int, seed: int):
    rng = np.random.default_rng(seed)
    nodes = list(topology.nodes())
    paths = []
    attempts = 0
    max_attempts = 20 * max(1, num_paths)
    while len(paths) < num_paths and attempts < max_attempts:
        attempts += 1
        src, dst = rng.choice(nodes, 2, replace=False)
        try:
            p = nx.shortest_path(topology, int(src), int(dst), weight='distance')
        except nx.NetworkXNoPath:
            continue
        if len(p) < 2:
            continue
        if p not in paths:
            paths.append([int(x) for x in p])
    if len(paths) < num_paths:
        raise RuntimeError(f'Paper8: could not find {num_paths} unique paths (found {len(paths)}).')
    return paths

def _compositions(total: int, parts: int, limit: int = 2500):
    out = []
    def rec(remaining: int, k: int, prefix):
        if len(out) >= limit:
            return
        if k == 1:
            out.append(prefix + [remaining])
            return
        for x in range(remaining + 1):
            if len(out) >= limit:
                return
            rec(remaining - x, k - 1, prefix + [x])
    rec(int(total), int(parts), [])
    return out

def generate_paper8_allocation_contexts(paths, qubit_cap, max_contexts_per_path: int = 2500):
    qubit_cap = list(qubit_cap)
    if len(qubit_cap) != len(paths):
        raise ValueError(f'Paper8: qubit_cap length {len(qubit_cap)} != paths length {len(paths)}')
    contexts = []
    for cap, path in zip(qubit_cap, paths):
        hops = max(1, len(path) - 1)
        comps = _compositions(int(cap), int(hops), limit=int(max_contexts_per_path))
        contexts.append(np.array(comps, dtype=int))
    return contexts

def get_physics_params(physics_model: str, current_frames: int, base_seed: int, qubit_cap):
    if physics_model != 'paper8':
        raise ValueError('This notebook only supports physics_model=paper8')

    p8 = FRAMEWORK_CONFIG['paper8']
    topo = Paper8RandomConnectedTopologyGenerator(
        num_nodes=p8.get('num_nodes', 50),
        connection_prob=p8.get('connection_prob', 0.08),
        seed=base_seed,
        fidelity_range=p8.get('fidelity_range', (0.6, 0.99)),
        rate_range=p8.get('rate_range', (0.7, 1.0)),
        pur_round_range=p8.get('pur_round_range', (0, 5)),
        swap_success_range=p8.get('swap_success_range', (0.7, 0.99)),
    ).generate()

    num_paths = int(p8.get('num_paths', len(qubit_cap)))
    paths = generate_paper8_paths(topo, num_paths=num_paths, seed=base_seed)
    contexts = generate_paper8_allocation_contexts(paths, qubit_cap, max_contexts_per_path=2500)

    noise_model = Paper8NoiseModel(topology=topo, paths=paths)
    fidelity_calc = Paper8FidelityCalculator(min_fidelity=float(p8.get('min_fidelity', 0.0)))

    print(f'📊 Paper8 Topology: {topo.number_of_nodes()} nodes, {topo.number_of_edges()} edges')
    print(f'📊 Paper8 Paths: {len(paths)}')

    return {
        'noise_model': noise_model,
        'fidelity_calculator': fidelity_calc,
        'external_topology': topo,
        'external_contexts': contexts,
        'external_rewards': None,
    }


## Run (AllocatorRunner)


In [ ]:
# --- Run: AllocatorRunner (Paper 8) ---
PHYSICS_MODELS = ['paper8']

for allocator_type in ALLOCATORS:
    print('\n' + '='*70)
    print(f'RUNNING: {allocator_type} on Paper 8')
    print('='*70)

    for scale in SCALES:
        print('\n' + '-'*70)
        print(f'Preparing: {allocator_type} at scale {scale}')
        print('-'*70)

        for physics_model in PHYSICS_MODELS:
            custom_config = ExperimentConfiguration(
                env_type=FRAMEWORK_CONFIG['main_env'],
                scenarios=test_scenarios,
                use_last_backup=True,
                models=models,
                attack_intensity=ATTACK_INTENSITY,
                scale=scale,
                base_capacity=True,
                overwrite=False,
            )

            alloc_runner = AllocatorRunner(
                allocator_type=allocator_type,
                physics_models=[physics_model],
                framework_config=FRAMEWORK_CONFIG,
                scales=[scale],
                runs=RUNS,
                models=models,
                test_scenarios=test_scenarios,
                config=custom_config,
            )

            alloc_runner.run(get_physics_params_func=get_physics_params)

print('
✓ Done')
